### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  # --multirun

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

---
## Google Cloud VM Setup (Alternative to Colab GPU Runtime)

There are two ways to use a GCP GPU VM for this project:

### Option A: SSH directly to the VM (Recommended)
This is simpler and more reliable for long training runs — no risk of Colab sessions timing out.

**One-time VM setup (run these in your GCP VM terminal via SSH):**
```bash
# 1. Clone the repo
git clone https://github.com/facebookresearch/emg2qwerty.git
cd emg2qwerty

# 2. Install dependencies
pip install -r requirements.txt

# 3. Copy your dataset into the data/ directory
#    (use gcloud storage cp, scp, or gdrive CLI)
mkdir -p data
# e.g.: gcloud storage cp gs://your-bucket/emg_data/*.hdf5 data/

# 4. (Optional) Start a tmux/screen session so training survives SSH disconnect
tmux new -s training
```

**Training commands to run on the VM:**
```bash
# TDS-Conv model (original)
python -m emg2qwerty.train user="single_user" trainer.accelerator=gpu trainer.devices=1 trainer.max_epochs=40

# LSTM model (new)
python -m emg2qwerty.train user="single_user" model=lstm_ctc trainer.accelerator=gpu trainer.devices=1 trainer.max_epochs=40
```

### Option B: Connect Colab to GCP VM as a custom runtime
Run the following on your VM over SSH, then connect Colab via **Runtime → Connect to a custom GCE VM**  
(requires your VM to have the **Colab Enterprise** runtime installed, available via Vertex AI Workbench).  
Alternatively, for standard VMs:
```bash
# On the VM — expose Jupyter so Colab can connect
pip install jupyter
jupyter notebook \
  --NotebookApp.allow_origin='https://colab.research.google.com' \
  --port=8888 --no-browser --ip=0.0.0.0
```
Then in Colab: **Runtime → Connect to a local runtime** and enter  
`http://<YOUR_VM_EXTERNAL_IP>:8888/?token=<token_from_terminal>`  
(Make sure firewall rule allows TCP port 8888 from your IP.)

> **Recommendation:** Use **Option A (SSH)** for training. It is more stable and does not depend on keeping a browser tab open.

---
## LSTM Model Experiments

A bidirectional LSTM encoder (`LSTMCTCModule`) has been added as an alternative to the TDS-Conv model.  
Architecture: `SpectrogramNorm → MultiBandRotationInvariantMLP → Flatten → BiLSTM → Linear → LogSoftmax`

Default config (`config/model/lstm_ctc.yaml`):
- `hidden_size: 512` — 512 units per direction (1024 total output)
- `num_layers: 3`
- `dropout: 0.1`
- `bidirectional: true`

### LSTM Training
- Checkpoints are saved under `logs/<date>/<time>/checkpoints/`.

In [ ]:
# Single-user LSTM training
!python -m emg2qwerty.train \
  model=lstm_ctc \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  trainer.max_epochs=40
  # --multirun

### LSTM Testing

- Replace `Your_Path_to_LSTM_Checkpoint` with the checkpoint path from the training run above.
- This prints both **val/CER** and **test/CER** (and their corresponding losses) to stdout.

In [ ]:
# Single-user LSTM testing (greedy CTC decoding)
!python -m emg2qwerty.train \
  model=lstm_ctc \
  user="single_user" \
  checkpoint="Your_Path_to_LSTM_Checkpoint" \
  train=False \
  trainer.accelerator=gpu \
  decoder=ctc_greedy
  # --multirun

### Comparing TDS-Conv vs LSTM

Run the cell below after both models have been trained to print a side-by-side summary of their val/CER and test/CER.  
Replace the checkpoint paths accordingly.

In [ ]:
import subprocess, json, re

def run_test(model, checkpoint):
    cmd = [
        "python", "-m", "emg2qwerty.train",
        f"model={model}",
        "user=single_user",
        f"checkpoint={checkpoint}",
        "train=False",
        "trainer.accelerator=gpu",
        "decoder=ctc_greedy",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout + result.stderr

tds_ckpt  = "Your_Path_to_TDS_Checkpoint"   # <-- replace
lstm_ckpt = "Your_Path_to_LSTM_Checkpoint"  # <-- replace

print("=== TDS-Conv Results ===")
print(run_test("tds_conv_ctc", tds_ckpt))
print("=== LSTM Results ===")
print(run_test("lstm_ctc", lstm_ckpt))